# RoseTTAFold2

Предсказание третичной структуры заданной аминокислотной последовательности с помощью RoseTTAFold2 в отдельном окружении Python 3.10.


In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir("/content")
if not Path("/content/bin/micromamba").exists():
    !curl -fL --retry 3 https://micro.mamba.pm/api/micromamba/linux-64/latest -o /content/micromamba.tar.bz2
    !tar -xjf /content/micromamba.tar.bz2 bin/micromamba

if not Path("/content/rf2_env/bin/python").exists():
    !MAMBA_ROOT_PREFIX=/content/micromamba /content/bin/micromamba create -y -p /content/rf2_env --override-channels -c pytorch -c nvidia -c conda-forge -c bioconda -c pyg -c dglteam/label/cu121 python=3.10 numpy=1.26 pytorch=2.2 pytorch-cuda=12.1 dgl=2.0.0.cu121 pyg hhsuite

!/content/rf2_env/bin/python -m pip install torchdata==0.7.1 matplotlib
Path("/content/variant39_rf2_driver.py").write_text('#@title setup RoseTTAFold2 (~1m)\nparams = "RF2_apr23" # @param ["RF2_apr23","RF2_jan24"]\n\nimport os, time, sys\nos.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"\n\nif params == "RF2_jan24" and not os.path.isfile(f"{params}.tgz"):\n  # send param download into background\n  os.system("(apt-get install aria2; aria2c -q -x 16 https://files.ipd.uw.edu/dimaio/RF2_jan24.tgz) &")\n\nif params == "RF2_apr23" and not os.path.isfile(f"{params}.tgz"):\n  # send param download into background\n  os.system("(apt-get install aria2; aria2c -q -x 16 https://files.ipd.uw.edu/dimaio/RF2_apr23.tgz) &")\n\nif not os.path.isdir("RoseTTAFold2"):\n  print("install RoseTTAFold2")\n  os.system("git clone https://github.com/uw-ipd/RoseTTAFold2.git")\n  os.system("pip install py3Dmol")\n\n  # 17Mar2024: adding --no-dependencies to avoid installing nvidia-cuda-* dependencies\n  # 25Aug2025: updating dgi install to work with latest pytorch\n  os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")\n  os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")\n  os.system("cd RoseTTAFold2/SE3Transformer; pip install .")\n\n  os.system("wget https://raw.githubusercontent.com/sokrypton/ColabFold/main/colabfold/colabfold.py -O colabfold_utils.py")\n\n  #os.system("pip install dgl -f https://data.dgl.ai/wheels/cu121/repo.html")\n  #os.system("cd RoseTTAFold2/SE3Transformer; pip -q install --no-cache-dir -r requirements.txt; pip -q install .")\n  #os.system("wget https://raw.githubusercontent.com/sokrypton/ColabFold/beta/colabfold/mmseqs/api.py")\n\n  # install hhsuite\n  print("install hhsuite")\n  os.makedirs("hhsuite", exist_ok=True)\n  os.system(f"curl -fsSL https://github.com/soedinglab/hh-suite/releases/download/v3.3.0/hhsuite-3.3.0-SSE2-Linux.tar.gz | tar xz -C hhsuite/")\n\nif not os.path.isfile(f"{params}.pt"):\n  time.sleep(5)\n\nif os.path.isfile(f"{params}.tgz.aria2"):\n  print("downloading RoseTTAFold2 params")\n  while os.path.isfile(f"{params}.tgz.aria2"):\n    time.sleep(5)\n\nif not os.path.isfile(f"{params}.pt"):\n  os.system(f"tar -zxvf {params}.tgz")\n  if params == "RF2_apr23":\n    os.system(f"mv weights/{params}.pt .")\n\nif not "IMPORTED" in dir():\n  if \'RoseTTAFold2/network\' not in sys.path:\n    os.environ["DGLBACKEND"] = "pytorch"\n    sys.path.append(\'RoseTTAFold2/network\')\n  if "hhsuite" not in os.environ[\'PATH\']:\n    os.environ[\'PATH\'] += ":hhsuite/bin:hhsuite/scripts"\n\n  import matplotlib.pyplot as plt\n  import numpy as np\n  from parsers import parse_a3m\n  #from api import run_mmseqs2\n  from colabfold_utils import run_mmseqs2\n  import py3Dmol\n  import torch\n  from string import ascii_uppercase, ascii_lowercase\n  import hashlib, re, os\n  import random\n\n  def get_hash(x): return hashlib.sha1(x.encode()).hexdigest()\n  alphabet_list = list(ascii_uppercase+ascii_lowercase)\n  from collections import OrderedDict, Counter\n\n  IMPORTED = True\n\nif not "pred" in dir() or params_sele != params:\n  from predict import Predictor\n  print("compile RoseTTAFold2")\n\n  if (torch.cuda.is_available()):\n    pred = Predictor(f"{params}.pt", torch.device("cuda:0"))\n  else:\n    print ("WARNING: using CPU")\n    pred = Predictor(f"{params}.pt", torch.device("cpu"))\n  params_sele = params\n\ndef get_unique_sequences(seq_list):\n    unique_seqs = list(OrderedDict.fromkeys(seq_list))\n    return unique_seqs\n\ndef run_mmseqs2_wrapper(*args, **kwargs):\n  kwargs[\'user_agent\'] = "colabfold/rosettafold2"\n  return run_mmseqs2(*args, **kwargs)\n\ndef get_msa(seq, jobname, cov=50, id=90, max_msa=2048,\n            mode="unpaired_paired"):\n\n  assert mode in ["unpaired","paired","unpaired_paired"]\n  seqs = [seq] if isinstance(seq,str) else seq\n\n  # collapse homooligomeric sequences\n  counts = Counter(seqs)\n  u_seqs = list(counts.keys())\n  u_nums = list(counts.values())\n\n  # expand homooligomeric sequences\n  first_seq = "/".join(sum([[x]*n for x,n in zip(u_seqs,u_nums)],[]))\n  msa = [first_seq]\n\n  path = os.path.join(jobname,"msa")\n  os.makedirs(path, exist_ok=True)\n  if mode in ["paired","unpaired_paired"] and len(u_seqs) > 1:\n    print("getting paired MSA")\n    out_paired = run_mmseqs2_wrapper(u_seqs, f"{path}/", use_pairing=True)\n    headers, sequences = [],[]\n    for a3m_lines in out_paired:\n      n = -1\n      for line in a3m_lines.split("\\n"):\n        if len(line) > 0:\n          if line.startswith(">"):\n            n += 1\n            if len(headers) < (n + 1):\n              headers.append([])\n              sequences.append([])\n            headers[n].append(line)\n          else:\n            sequences[n].append(line)\n    # filter MSA\n    with open(f"{path}/paired_in.a3m","w") as handle:\n      for n,sequence in enumerate(sequences):\n        handle.write(f">n{n}\\n{\'\'.join(sequence)}\\n")\n    os.system(f"hhfilter -i {path}/paired_in.a3m -id {id} -cov {cov} -o {path}/paired_out.a3m")\n    with open(f"{path}/paired_out.a3m","r") as handle:\n      for line in handle:\n        if line.startswith(">"):\n          n = int(line[2:])\n          xs = sequences[n]\n          # expand homooligomeric sequences\n          xs = [\'/\'.join([x]*num) for x,num in zip(xs,u_nums)]\n          msa.append(\'/\'.join(xs))\n\n  if len(msa) < max_msa and (mode in ["unpaired","unpaired_paired"] or len(u_seqs) == 1):\n    print("getting unpaired MSA")\n    out = run_mmseqs2_wrapper(u_seqs,f"{path}/")\n    Ls = [len(seq) for seq in u_seqs]\n    sub_idx = []\n    sub_msa = []\n    sub_msa_num = 0\n    for n,a3m_lines in enumerate(out):\n      sub_msa.append([])\n      with open(f"{path}/in_{n}.a3m","w") as handle:\n        handle.write(a3m_lines)\n      # filter\n      os.system(f"hhfilter -i {path}/in_{n}.a3m -id {id} -cov {cov} -o {path}/out_{n}.a3m")\n      with open(f"{path}/out_{n}.a3m","r") as handle:\n        for line in handle:\n          if not line.startswith(">"):\n            xs = [\'-\'*l for l in Ls]\n            xs[n] = line.rstrip()\n            # expand homooligomeric sequences\n            xs = [\'/\'.join([x]*num) for x,num in zip(xs,u_nums)]\n            sub_msa[-1].append(\'/\'.join(xs))\n            sub_msa_num += 1\n      sub_idx.append(list(range(len(sub_msa[-1]))))\n\n    while len(msa) < max_msa and sub_msa_num > 0:\n      for n in range(len(sub_idx)):\n        if len(sub_idx[n]) > 0:\n          msa.append(sub_msa[n][sub_idx[n].pop(0)])\n          sub_msa_num -= 1\n        if len(msa) == max_msa:\n          break\n\n  with open(f"{jobname}/msa.a3m","w") as handle:\n    for n,sequence in enumerate(msa):\n      handle.write(f">n{n}\\n{sequence}\\n")\n#@title ###run RoseTTAFold2\nsequence = "MENEDGYMTLSFKNRCKSKQKSKDFSLYPQYYCLLLIFGCIVILIFIMTGIDLKFWHKKMDFSQNVNVSSLSGHNYLCPNDWLLNEGKCYWFSTSFKTWK" #@param {type:"string"}\njobname = "variant39_rf2" #@param {type:"string"}\n\n#@markdown symmetry settings\nsym = "X" #@param ["X","C", "D", "T", "I", "O"]\norder = 1 #@param ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"] {type:"raw"}\nmsa_concat_mode = "diag" #@param ["diag", "repeat", "default"]\n\n#@markdown msa settings\nmsa_method = "mmseqs2" #@param ["mmseqs2","single_sequence","custom_a3m"]\npair_mode = "unpaired_paired" #@param ["unpaired_paired","paired","unpaired"] {type:"string"}\ncollapse_identical = False #@param {type:"boolean"}\n\n#@markdown RoseTTAFold2 settings\nnum_recycles = 6 #@param [0, 1, 3, 6, 12, 24] {type:"raw"}\n\n#@markdown stochastic settings\nuse_mlm = False #@param {type:"boolean"}\nuse_dropout = False #@param {type:"boolean"}\nmax_msa = 256 #@param [16, 32, 64, 128, 256, 512] {type:"raw"}\nrandom_seed = 0 #@param {type:"integer"}\nnum_models = 1 #@param ["1", "5", "10", "15", "20", "25"] {type:"raw"}\n\n# process\nmax_extra_msa = max_msa * 8\n\nsequence = re.sub("[^A-Z:]", "", sequence.replace("/",":").upper())\nsequence = re.sub(":+",":",sequence)\nsequence = re.sub("^[:]+","",sequence)\nsequence = re.sub("[:]+$","",sequence)\n\nif sym in ["X","C"]:\n  copies = order\nelif sym in ["D"]:\n  copies = order * 2\nelse:\n  copies = {"T":12,"O":24,"I":60}[sym]\n  order = ""\nsymm = sym + str(order)\n\nsequences = sequence.replace(":","/").split("/")\nif collapse_identical:\n  u_sequences = get_unique_sequences(sequences)\nelse:\n  u_sequences = sequences\nsequences = sum([u_sequences] * copies,[])\nlengths = [len(s) for s in sequences]\n\n# TODO\n#subcrop = 1000 if sum(lengths) > 1400 else -1\n\nsubcrop = -1\ntopk = 1536\n\n\nsequence = "/".join(sequences)\njobname = jobname+"_"+symm+"_"+get_hash(sequence)[:5]\n\nprint(f"jobname: {jobname}")\nprint(f"lengths: {lengths}")\n\nos.makedirs(jobname, exist_ok=True)\nif msa_method == "mmseqs2":\n  get_msa(u_sequences, jobname, mode=pair_mode, max_msa=max_extra_msa)\n\nelif msa_method == "single_sequence":\n  u_sequence = "/".join(u_sequences)\n  with open(f"{jobname}/msa.a3m","w") as a3m:\n    a3m.write(f">{jobname}\\n{u_sequence}\\n")\n\nelif msa_method == "custom_a3m":\n  print("upload custom a3m")\n  msa_dict = files.upload()\n  lines = msa_dict[list(msa_dict.keys())[0]].decode().splitlines()\n  a3m_lines = []\n  for line in lines:\n    line = line.replace("\\x00","")\n    if len(line) > 0 and not line.startswith(\'#\'):\n      a3m_lines.append(line)\n\n  with open(f"{jobname}/msa.a3m","w") as a3m:\n    a3m.write("\\n".join(a3m_lines))\n\nbest_plddt = None\nbest_seed = None\nfor seed in range(random_seed,random_seed+num_models):\n  torch.manual_seed(seed)\n  random.seed(seed)\n  np.random.seed(seed)\n  npz = f"{jobname}/rf2_seed{seed}_00.npz"\n  pred.predict(inputs=[f"{jobname}/msa.a3m"],\n               out_prefix=f"{jobname}/rf2_seed{seed}",\n               symm=symm,\n               ffdb=None, #TODO (templates),\n               n_recycles=num_recycles,\n               msa_mask=0.15 if use_mlm else 0.0,\n               msa_concat_mode=msa_concat_mode,\n               nseqs=max_msa,\n               nseqs_full=max_extra_msa,\n               subcrop=subcrop,\n               topk=topk,\n               is_training=use_dropout)\n  plddt = np.load(npz)["lddt"].mean()\n  if best_plddt is None or plddt > best_plddt:\n    best_plddt = plddt\n    best_seed = seed\n#@title Download prediction\n\n#@markdown Once this cell has been executed, a zip-archive with\n#@markdown the obtained prediction will be automatically downloaded\n#@markdown to your computer.\n\n# add settings file\nsettings_path = f"{jobname}/settings.txt"\nwith open(settings_path, "w") as text_file:\n  text_file.write(f"method=RoseTTAFold2\\n")\n  text_file.write(f"params={params}\\n")\n  text_file.write(f"sequence={sequence}\\n")\n  text_file.write(f"sym={sym}\\n")\n  text_file.write(f"order={order}\\n")\n  text_file.write(f"msa_concat_mode={msa_concat_mode}\\n")\n  text_file.write(f"collapse_identical={collapse_identical}")\n  text_file.write(f"random_seed={random_seed}\\n")\n  text_file.write(f"msa_method={msa_method}\\n")\n  text_file.write(f"num_recycles={num_recycles}\\n")\n  text_file.write(f"use_mlm={use_mlm}\\n")\n  text_file.write(f"use_dropout={use_dropout}\\n")\n  text_file.write(f"num_models={num_models}\\n")\n\n# --- Download the predictions ---\nos.system(f"zip -r {jobname}.zip {jobname}")\n', encoding="utf-8")
env = os.environ.copy()
env["PATH"] = "/content/rf2_env/bin:" + env["PATH"]
env["DGLBACKEND"] = "pytorch"
log_path = "/content/variant39_rf2_driver.log"
with open(log_path, "w") as log:
    result = subprocess.run(["/content/rf2_env/bin/python", "/content/variant39_rf2_driver.py"], cwd="/content", env=env, stdout=log, stderr=subprocess.STDOUT)
log_lines = Path(log_path).read_text().splitlines()
print("
".join(log_lines[-100:]))
if result.returncode != 0:
    raise RuntimeError(f"RoseTTAFold2 driver failed. Full log: {log_path}")


In [ ]:
import hashlib
import numpy as np
import py3Dmol
from google.colab import files

sequence = "MENEDGYMTLSFKNRCKSKQKSKDFSLYPQYYCLLLIFGCIVILIFIMTGIDLKFWHKKMDFSQNVNVSSLSGHNYLCPNDWLLNEGKCYWFSTSFKTWK"
jobname = "variant39_rf2_X1_" + hashlib.sha1(sequence.encode()).hexdigest()[:5]
pdb_path = f"{jobname}/rf2_seed0_00_pred.pdb"
metrics = np.load(f"{jobname}/rf2_seed0_00.npz")
print("Mean pLDDT:", float(metrics["lddt"].mean()))
view = py3Dmol.view()
with open(pdb_path) as f:
    view.addModel(f.read(), "pdb")
view.setStyle({'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'roygb', 'min': 50, 'max': 90}}})
view.zoomTo()
view.show()
from zipfile import ZipFile
with ZipFile(f"{jobname}.zip", "a") as archive:
    archive.write("/content/variant39_rf2_driver.log", "driver.log")
files.download(f"{jobname}.zip")
